In [2]:
# STEP3.2_07_thesis_cross_plot_v2.py
#
# 目的: 「Sample Aのエリート」が「Sample Bの地図」上でどう分布するかを可視化する
# 修正: ファイルパスを絶対パスで正しく指定するように変更
#

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 設定 (Configuration)
# ==========================================
RUN_ID = "20251130_153500"
ROOT_BASE = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

# 入力ディレクトリの定義 (ここが修正ポイント)
# 1. MDS座標+PCEデータ (STEP 05の出力先)
THESIS_DATA_DIR = os.path.join(ROOT_BASE, "Thesis_Figures", f"run_{RUN_ID}")

# 2. クラスタリングラベル (STEP 04の出力先)
#    注意: unit="samples" のフォルダの下にある
SUMMARY_DIR = os.path.join(ROOT_BASE, "sub", "04_summary_STEP3.2_signlessCorr", f"run_{RUN_ID}", "samples")

# 出力先
OUTPUT_DIR = os.path.join(ROOT_BASE, "Thesis_Figures", f"run_{RUN_ID}", "Cross_Validation")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# プロット設定
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Helvetica']
plt.rcParams['font.size'] = 14
plt.rcParams['figure.dpi'] = 300

# ==========================================
# 2. ファイルパスの構築
# ==========================================

# A (OH) 用ファイル
file_thesis_A = os.path.join(THESIS_DATA_DIR, "ThesisData_A_OH_plus_others_MDS_Targets.csv")
file_labels_A = os.path.join(SUMMARY_DIR, "cluster_labels_matrix_samples_A_OH_plus_others_20251130_153500.csv")

# B (FP) 用ファイル
file_thesis_B = os.path.join(THESIS_DATA_DIR, "ThesisData_B_FP_plus_others_MDS_Targets.csv")
file_labels_B = os.path.join(SUMMARY_DIR, "cluster_labels_matrix_samples_B_FP_plus_others_20251130_153500.csv")

# 存在チェック
for f in [file_thesis_A, file_labels_A, file_thesis_B, file_labels_B]:
    if not os.path.exists(f):
        print(f"[ERROR] File not found: {f}")

# ==========================================
# 3. データ読み込み & ID抽出
# ==========================================

def load_and_merge(path_thesis, path_labels):
    if not os.path.exists(path_thesis) or not os.path.exists(path_labels):
        return None
        
    df_t = pd.read_csv(path_thesis)
    df_l = pd.read_csv(path_labels)
    
    # ID列の統一 (R由来の Unnamed: 0 を ID に)
    if 'Unnamed: 0' in df_t.columns:
        df_t.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
    
    # 結合
    df = pd.merge(df_t, df_l, on='ID', how='inner')
    return df

print("Loading Data...")
df_A = load_and_merge(file_thesis_A, file_labels_A)
df_B = load_and_merge(file_thesis_B, file_labels_B)

if df_A is None or df_B is None:
    raise FileNotFoundError("Some input files are missing. Please check the paths above.")

# エリートクラスターのIDを特定 (前回の解析に基づく)
# A (OH): k=50 の Cluster 19
best_cluster_A = 19
col_A = 'linear_top3_k50'
ids_elite_A = df_A[df_A[col_A] == best_cluster_A]['ID'].values

# B (FP): k=50 の Cluster 18
best_cluster_B = 18
col_B = 'linear_top3_k50'
ids_elite_B = df_B[df_B[col_B] == best_cluster_B]['ID'].values

print(f"Elite IDs in A (Cluster {best_cluster_A}): {len(ids_elite_A)}")
print(f"Elite IDs in B (Cluster {best_cluster_B}): {len(ids_elite_B)}")

# ==========================================
# 4. クロスプロット作成関数
# ==========================================

def plot_cross_highlight(df_map, highlight_ids, map_name, source_name, output_tag):
    """
    df_map: 地図として使うデータフレーム (V1, V2を持つ)
    highlight_ids: 赤く光らせたいIDのリスト
    map_name: 地図の名前 (例: "Sample B (FP)")
    source_name: IDの出所 (例: "Sample A's Elite")
    """
    
    # ハイライト対象かどうか
    df_map = df_map.copy()
    df_map['is_highlight'] = df_map['ID'].isin(highlight_ids)
    
    fig, ax = plt.subplots(figsize=(7, 7))
    
    # 1. 背景（その他大勢）
    bg = df_map[~df_map['is_highlight']]
    ax.scatter(bg['V1'], bg['V2'], c='lightgray', s=30, alpha=0.5, label='Others')
    
    # 2. ハイライト（エリート）
    hl = df_map[df_map['is_highlight']]
    
    if len(hl) > 0:
        ax.scatter(hl['V1'], hl['V2'], c='red', s=60, edgecolors='black', linewidth=0.8, 
                   label=f"{source_name}\n(n={len(hl)})")
    else:
        # 念のため0個の場合の処理
        print(f"[WARN] No matching IDs found for {source_name} on {map_name}")

    ax.set_title(f"Projection on {map_name}", fontweight="bold")
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.legend(loc='best')
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"CrossPlot_{output_tag}.png"))
    plt.savefig(os.path.join(OUTPUT_DIR, f"CrossPlot_{output_tag}.pdf"))
    plt.close()
    print(f"-> Plot Saved: CrossPlot_{output_tag}")

# ==========================================
# 5. 実行: 相互投影
# ==========================================

print("\n--- Generating Cross-Validation Plots ---")

# ケース1: Aのエリート(OH) を Bの地図(FP) に投影
# 仮説: バラバラになるはず
plot_cross_highlight(df_B, ids_elite_A, 
                     map_name="Structure Space (FP)", 
                     source_name="Material Elites (OH)", 
                     output_tag="A_on_B")

# ケース2: Bのエリート(FP) を Aの地図(OH) に投影
# 仮説: バラバラになるはず
plot_cross_highlight(df_A, ids_elite_B, 
                     map_name="Material Space (OH)", 
                     source_name="Structure Elites (FP)", 
                     output_tag="B_on_A")

# ケース3 (参照用): Aのエリートを Aの地図に表示 (当然固まっている)
plot_cross_highlight(df_A, ids_elite_A, 
                     map_name="Material Space (OH)", 
                     source_name="Material Elites (OH)", 
                     output_tag="A_on_A_Reference")

print("\n✅ Done! Check 'Thesis_Figures/.../Cross_Validation' folder.")

Loading Data...
Elite IDs in A (Cluster 19): 11
Elite IDs in B (Cluster 18): 20

--- Generating Cross-Validation Plots ---
-> Plot Saved: CrossPlot_A_on_B
-> Plot Saved: CrossPlot_B_on_A
-> Plot Saved: CrossPlot_A_on_A_Reference

✅ Done! Check 'Thesis_Figures/.../Cross_Validation' folder.
